# TransformerLens Multilingual LLM Evaluation

This notebook extends the accuracy evaluation in `step1.py` with mechanistic interpretability: using TransformerLens on Mistral-7B to examine *how* the model processes English, Spanish, and Basque internally.

**Two analyses:**
1. **Logit Lens** — per-layer predictions to see when the model converges on the correct answer, and whether it does so faster for English than Basque
2. **Representation Similarity** — cosine similarity of residual stream activations for the same question across languages at each layer

**Run on Google Colab with a T4 GPU.**

## Cell 1 — Install & Imports

In [ ]:
# ── Step 1: Install packages ──────────────────────────────────────────────────
# transformer-lens 2.17.0 requires numpy<2 and beartype<0.15, so we must NOT
# upgrade those. The --force-reinstall on numpy repairs binary corruption from
# any prior upgrade attempts.
#
# After running this cell: Runtime > Restart session, then run from Step 2 onward.

!pip install numpy==1.26.4 --force-reinstall --quiet
!pip install transformer_lens circuitsvis einops jaxtyping plotly --quiet

print("Installs complete.")
print("Now restart the runtime: Runtime > Restart session  (or Ctrl+M .)")
print("Then run all cells from Step 2 onward.")

In [ ]:
# ── Step 2: Imports (run after restarting the runtime) ───────────────────────
import transformer_lens
from transformer_lens import HookedTransformer
import torch
import json
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import gc

# Colab plotly renderer
import plotly.io as pio
pio.renderers.default = "colab"

print(f"TransformerLens version: {transformer_lens.__version__}")
print(f"NumPy version:           {np.__version__}")
print(f"PyTorch version:         {torch.__version__}")
print(f"CUDA available:          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2 — Load Data

In [ ]:
# Mount Google Drive if running in Colab, or use local paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust DATA_DIR to your Drive path if needed
    DATA_DIR = Path('/content/drive/MyDrive/Kai-Justin-NLP-Project/data')
    if not DATA_DIR.exists():
        # Fall back to uploading files directly
        DATA_DIR = Path('/content/data')
        DATA_DIR.mkdir(exist_ok=True)
        print("Drive path not found. Upload data files or adjust DATA_DIR.")
except ImportError:
    # Running locally
    DATA_DIR = Path('data')

print(f"Using data directory: {DATA_DIR}")

def load_jsonl(path):
    items = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                item = json.loads(line)
                items[item['id']] = item
    return items

english_data = load_jsonl(DATA_DIR / 'english.jsonl')
spanish_data = load_jsonl(DATA_DIR / 'spanish.jsonl')
basque_data  = load_jsonl(DATA_DIR / 'basque.jsonl')

print(f"Loaded: {len(english_data)} English, {len(spanish_data)} Spanish, {len(basque_data)} Basque")

# Align by id — keep only ids present in all three
common_ids = set(english_data.keys()) & set(spanish_data.keys()) & set(basque_data.keys())
print(f"Matched triplets: {len(common_ids)}")

triplets = {
    qid: {
        'english': english_data[qid],
        'spanish': spanish_data[qid],
        'basque':  basque_data[qid],
    }
    for qid in sorted(common_ids)
}

# Preview
sample_id = sorted(common_ids)[0]
print(f"\nSample triplet (id={sample_id}):")
for lang in ['english', 'spanish', 'basque']:
    print(f"  [{lang}] {triplets[sample_id][lang]['question']}")
    print(f"          ground_truth: {triplets[sample_id][lang]['ground_truth']}")

## Cell 3 — Load Model

Loads Mistral-7B-v0.1 in float16. If you hit OOM on a T4 (15GB), fall back to `gpt2-medium` (355M params) to validate methodology — note it is English-only.

In [ ]:
# Set MODEL_NAME to 'mistral-7b-v0.1' for Mistral or 'gpt2-medium' for a lightweight fallback
MODEL_NAME = 'mistral-7b-v0.1'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading {MODEL_NAME} on {device} ...")

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    fold_ln=True,
    center_writing_weights=True,
    device=device,
)
model.eval()
torch.cuda.empty_cache()

print(f"Model loaded: {MODEL_NAME}")
print(f"  Layers:  {model.cfg.n_layers}")
print(f"  d_model: {model.cfg.d_model}")
print(f"  d_vocab: {model.cfg.d_vocab}")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM used: {used:.1f} / {total:.1f} GB")

## Cell 4 — Logit Lens Helper

In [ ]:
def logit_lens(model, tokens):
    """
    Return per-layer top-token probabilities for the last token position.

    Args:
        model: HookedTransformer
        tokens: (1, seq_len) int tensor

    Returns:
        list of n_layers tensors, each shape (vocab_size,), on CPU as float32
    """
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    n_layers = model.cfg.n_layers
    results = []
    with torch.no_grad():
        for layer in range(n_layers):
            residual = cache[f"blocks.{layer}.hook_resid_post"]  # (seq_len, d_model)
            # Project last token through final layer norm and unembed
            last = residual[-1:, :]  # (1, d_model)
            logits = model.unembed(model.ln_final(last))   # (1, vocab_size)
            probs = logits.softmax(dim=-1).squeeze().cpu().float()  # (vocab_size,)
            results.append(probs)
    del cache
    torch.cuda.empty_cache()
    return results  # list of n_layers tensors


def rank_of_token(probs, token_id):
    """Return the rank (0 = highest probability) of token_id in probs."""
    sorted_ids = probs.argsort(descending=True)
    rank = (sorted_ids == token_id).nonzero(as_tuple=True)[0].item()
    return rank


def entropy(probs):
    """Shannon entropy of a probability distribution."""
    p = probs.clamp(min=1e-12)
    return -(p * p.log()).sum().item()


# --- Sanity check ---
# Run logit lens on a simple English math question and confirm the correct answer
# token rises in probability across layers.
print("Running sanity check: 'What is 15+27?'")
test_question = "What is 15+27?"
test_tokens = model.to_tokens(test_question, prepend_bos=True)  # (1, seq_len)
print(f"  Sequence length: {test_tokens.shape[1]}")

test_results = logit_lens(model, test_tokens)

# The ground-truth answer is "42"; check its token id
gt_str = "42"
gt_token_id = model.to_single_token(" " + gt_str)  # space prefix for mid-sentence token
print(f"  Token id for ' {gt_str}': {gt_token_id}")

print("  Layer | Rank of '42' | Entropy")
for layer_idx, probs in enumerate(test_results):
    r = rank_of_token(probs, gt_token_id)
    e = entropy(probs)
    if layer_idx % 4 == 0 or layer_idx == len(test_results) - 1:
        print(f"  {layer_idx:5d} | {r:12d} | {e:.3f}")

print("Sanity check complete.")

## Cell 5 — Logit Lens Analysis

For each of the 60 question triplets, tokenize each language's question and run `logit_lens()`. Track rank of ground-truth token and entropy at each layer.

**Note**: Run on 5 questions first (`PILOT_RUN = True`) to validate, then set `PILOT_RUN = False` for the full 60.

In [ ]:
PILOT_RUN = True   # Set to False after validating on 5 questions
LANGUAGES = ['english', 'spanish', 'basque']

all_ids = sorted(triplets.keys())
if PILOT_RUN:
    run_ids = all_ids[:5]
    print(f"PILOT RUN: processing {len(run_ids)} questions")
else:
    run_ids = all_ids
    print(f"FULL RUN: processing {len(run_ids)} questions")

# Results structure: {id: {lang: {layer: {rank_of_gt: int, entropy: float}}}}
logit_lens_results = {}

for i, qid in enumerate(run_ids):
    logit_lens_results[qid] = {}
    triplet = triplets[qid]
    category = triplet['english']['category']

    for lang in LANGUAGES:
        item = triplet[lang]
        question = item['question']
        gt = item['ground_truth'].strip()

        # Tokenize; try space-prefixed token first, then bare token
        try:
            gt_token_id = model.to_single_token(" " + gt)
        except Exception:
            try:
                gt_token_id = model.to_single_token(gt)
            except Exception:
                # Ground truth maps to multiple tokens; use the first one
                gt_tokens = model.to_tokens(gt, prepend_bos=False)
                gt_token_id = gt_tokens[0, 0].item()

        tokens = model.to_tokens(question, prepend_bos=True)

        # Shape check
        assert tokens.ndim == 2 and tokens.shape[0] == 1, f"Unexpected token shape: {tokens.shape}"

        layer_results = logit_lens(model, tokens)

        layer_data = {}
        for layer_idx, probs in enumerate(layer_results):
            r = rank_of_token(probs, gt_token_id)
            e = entropy(probs)
            layer_data[layer_idx] = {'rank_of_gt': r, 'entropy': e}

        logit_lens_results[qid][lang] = layer_data

    if (i + 1) % 5 == 0 or i == len(run_ids) - 1:
        print(f"  Processed {i + 1}/{len(run_ids)} — id={qid} [{category}]")
    gc.collect()
    torch.cuda.empty_cache()

print("\nLogit lens analysis complete.")

# Save to JSON
output_path = 'logit_lens_results.json'
with open(output_path, 'w') as f:
    json.dump(logit_lens_results, f)
print(f"Saved to {output_path}")

## Cell 6 — Logit Lens Plots

1. Mean rank of ground-truth token by layer — line plot per language, faceted by category
2. Mean entropy by layer — same structure
3. Layer of first correct top-1 prediction — histogram comparing English vs Basque

In [ ]:
# Optionally reload saved results
# with open('logit_lens_results.json') as f:
#     logit_lens_results = json.load(f)

n_layers = model.cfg.n_layers
categories = ['math', 'factual', 'reasoning']
lang_colors = {'english': '#1f77b4', 'spanish': '#ff7f0e', 'basque': '#2ca02c'}

# ── Build long-form data ──────────────────────────────────────────────────────
rows = []  # each row: {qid, category, lang, layer, rank_of_gt, entropy}
for qid, lang_data in logit_lens_results.items():
    category = triplets[qid]['english']['category']
    for lang, layer_data in lang_data.items():
        for layer_str, metrics in layer_data.items():
            rows.append({
                'qid': qid,
                'category': category,
                'lang': lang,
                'layer': int(layer_str),
                'rank_of_gt': metrics['rank_of_gt'],
                'entropy': metrics['entropy'],
            })

# ── Plot 1: Mean rank of ground-truth token by layer ──────────────────────────
fig1 = make_subplots(
    rows=1, cols=len(categories),
    subplot_titles=[f"{cat.capitalize()}" for cat in categories],
    shared_yaxes=True,
)

for col_idx, cat in enumerate(categories, start=1):
    cat_rows = [r for r in rows if r['category'] == cat]
    for lang in LANGUAGES:
        lang_rows = [r for r in cat_rows if r['lang'] == lang]
        # Mean rank per layer
        layer_to_ranks = {}
        for r in lang_rows:
            layer_to_ranks.setdefault(r['layer'], []).append(r['rank_of_gt'])
        layers_sorted = sorted(layer_to_ranks.keys())
        mean_ranks = [np.mean(layer_to_ranks[l]) for l in layers_sorted]
        fig1.add_trace(
            go.Scatter(
                x=layers_sorted, y=mean_ranks,
                mode='lines', name=lang.capitalize(),
                line=dict(color=lang_colors[lang]),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig1.update_layout(
    title='Mean Rank of Ground-Truth Token by Layer (lower = better)',
    height=400,
)
fig1.update_yaxes(title_text='Mean rank', row=1, col=1)
fig1.update_xaxes(title_text='Layer')
fig1.show()

# ── Plot 2: Mean entropy by layer ─────────────────────────────────────────────
fig2 = make_subplots(
    rows=1, cols=len(categories),
    subplot_titles=[f"{cat.capitalize()}" for cat in categories],
    shared_yaxes=True,
)

for col_idx, cat in enumerate(categories, start=1):
    cat_rows = [r for r in rows if r['category'] == cat]
    for lang in LANGUAGES:
        lang_rows = [r for r in cat_rows if r['lang'] == lang]
        layer_to_entropy = {}
        for r in lang_rows:
            layer_to_entropy.setdefault(r['layer'], []).append(r['entropy'])
        layers_sorted = sorted(layer_to_entropy.keys())
        mean_entropy = [np.mean(layer_to_entropy[l]) for l in layers_sorted]
        fig2.add_trace(
            go.Scatter(
                x=layers_sorted, y=mean_entropy,
                mode='lines', name=lang.capitalize(),
                line=dict(color=lang_colors[lang]),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig2.update_layout(
    title='Mean Entropy by Layer (lower = more confident)',
    height=400,
)
fig2.update_yaxes(title_text='Mean entropy', row=1, col=1)
fig2.update_xaxes(title_text='Layer')
fig2.show()

# ── Plot 3: Layer of first correct top-1 prediction ───────────────────────────
# For each (qid, lang), find the first layer where rank_of_gt == 0
first_correct = {}  # {qid: {lang: first_correct_layer or None}}
for qid, lang_data in logit_lens_results.items():
    first_correct[qid] = {}
    for lang, layer_data in lang_data.items():
        fc = None
        for layer_str in sorted(layer_data.keys(), key=int):
            if layer_data[layer_str]['rank_of_gt'] == 0:
                fc = int(layer_str)
                break
        first_correct[qid][lang] = fc

hist_data = {'english': [], 'basque': []}
for qid in logit_lens_results:
    for lang in ['english', 'basque']:
        fc = first_correct[qid].get(lang)
        if fc is not None:
            hist_data[lang].append(fc)

fig3 = go.Figure()
for lang in ['english', 'basque']:
    fig3.add_trace(go.Histogram(
        x=hist_data[lang],
        name=lang.capitalize(),
        opacity=0.7,
        marker_color=lang_colors[lang],
        xbins=dict(start=0, end=n_layers, size=1),
    ))

fig3.update_layout(
    title='Layer of First Correct Top-1 Prediction: English vs Basque',
    xaxis_title='Layer',
    yaxis_title='Count',
    barmode='overlay',
    height=400,
)
fig3.show()

# Summary stats
for lang in ['english', 'spanish', 'basque']:
    fc_vals = [first_correct[qid].get(lang) for qid in logit_lens_results if first_correct[qid].get(lang) is not None]
    never = sum(1 for qid in logit_lens_results if first_correct[qid].get(lang) is None)
    if fc_vals:
        print(f"{lang:8s}: mean first-correct layer = {np.mean(fc_vals):.1f}, "
              f"never correct = {never}/{len(logit_lens_results)}")

## Cell 7 — Representation Similarity Helper

In [ ]:
def get_residual_streams(model, tokens):
    """
    Return the residual stream at each layer for the last token position.

    Args:
        model: HookedTransformer
        tokens: (1, seq_len) int tensor

    Returns:
        list of n_layers tensors, each shape (d_model,), on CPU as float32
    """
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, remove_batch_dim=True)
    streams = []
    for layer in range(model.cfg.n_layers):
        r = cache[f"blocks.{layer}.hook_resid_post"]  # (seq_len, d_model)
        # Shape check
        assert r.shape[-1] == model.cfg.d_model, \
            f"Unexpected d_model: expected {model.cfg.d_model}, got {r.shape[-1]}"
        streams.append(r[-1].cpu().float())  # last token, (d_model,)
    del cache
    torch.cuda.empty_cache()
    return streams  # list of n_layers tensors of shape (d_model,)


def cosine_sim(a, b):
    """Cosine similarity between two 1D tensors."""
    return torch.nn.functional.cosine_similarity(
        a.unsqueeze(0), b.unsqueeze(0)
    ).item()


# ── Sanity check: English↔English similarity should be 1.0 at every layer ────
print("Sanity check: English↔English cosine similarity (should be 1.0 everywhere)")
sample_q = triplets[sorted(triplets.keys())[0]]['english']['question']
toks = model.to_tokens(sample_q, prepend_bos=True)
streams_a = get_residual_streams(model, toks)
streams_b = get_residual_streams(model, toks)
sims = [cosine_sim(streams_a[l], streams_b[l]) for l in range(len(streams_a))]
print(f"  Min: {min(sims):.6f}, Max: {max(sims):.6f}, Mean: {np.mean(sims):.6f}")
assert all(abs(s - 1.0) < 1e-3 for s in sims), "Self-similarity sanity check FAILED"
print("  Passed.")

## Cell 8 — Representation Similarity Analysis

For each question triplet, compute per-layer cosine similarity between residual stream activations across language pairs.

**Hypothesis**: similarity increases in deeper layers as the model moves from syntax/language-specific representations toward semantic/language-agnostic ones.

In [ ]:
PILOT_RUN_SIM = True   # Set to False for full 60-question run
LANG_PAIRS = [('english', 'spanish'), ('english', 'basque'), ('spanish', 'basque')]

run_ids_sim = all_ids[:5] if PILOT_RUN_SIM else all_ids
print(f"Running similarity analysis on {len(run_ids_sim)} questions")

# Results structure: {id: {pair_str: {layer: cosine_sim}}}
sim_results = {}

for i, qid in enumerate(run_ids_sim):
    triplet = triplets[qid]
    category = triplet['english']['category']

    # Compute residual streams for all 3 languages
    lang_streams = {}
    for lang in LANGUAGES:
        question = triplet[lang]['question']
        tokens = model.to_tokens(question, prepend_bos=True)
        lang_streams[lang] = get_residual_streams(model, tokens)

    # Per-layer cosine similarity for each pair
    sim_results[qid] = {}
    for lang_a, lang_b in LANG_PAIRS:
        pair_key = f"{lang_a}_{lang_b}"
        sim_results[qid][pair_key] = {}
        for layer in range(model.cfg.n_layers):
            s = cosine_sim(lang_streams[lang_a][layer], lang_streams[lang_b][layer])
            sim_results[qid][pair_key][layer] = s

    if (i + 1) % 5 == 0 or i == len(run_ids_sim) - 1:
        print(f"  Processed {i + 1}/{len(run_ids_sim)} — id={qid} [{category}]")
    gc.collect()
    torch.cuda.empty_cache()

print("\nRepresentation similarity analysis complete.")

# Save to JSON
sim_output_path = 'sim_results.json'
with open(sim_output_path, 'w') as f:
    # JSON keys must be strings
    serializable = {
        qid: {
            pair: {str(layer): v for layer, v in layer_data.items()}
            for pair, layer_data in pair_data.items()
        }
        for qid, pair_data in sim_results.items()
    }
    json.dump(serializable, f)
print(f"Saved to {sim_output_path}")

## Cell 9 — Representation Similarity Plots

1. Mean cosine similarity by layer — line plot per language pair, faceted by category
2. Layer where similarity first crosses 0.9 threshold — summary table
3. Heatmap: layer × category × language pair

In [ ]:
# Optionally reload
# with open('sim_results.json') as f:
#     sim_results_raw = json.load(f)
#     sim_results = {
#         qid: {pair: {int(l): v for l, v in ld.items()} for pair, ld in pd.items()}
#         for qid, pd in sim_results_raw.items()
#     }

SIMILARITY_THRESHOLD = 0.9
pair_labels = {
    'english_spanish': 'English ↔ Spanish',
    'english_basque':  'English ↔ Basque',
    'spanish_basque':  'Spanish ↔ Basque',
}
pair_colors = {
    'english_spanish': '#9467bd',
    'english_basque':  '#d62728',
    'spanish_basque':  '#8c564b',
}

# ── Build long-form sim rows ──────────────────────────────────────────────────
sim_rows = []  # {qid, category, pair, layer, cosine_sim}
for qid, pair_data in sim_results.items():
    category = triplets[qid]['english']['category']
    for pair, layer_data in pair_data.items():
        for layer, sim in layer_data.items():
            sim_rows.append({
                'qid': qid,
                'category': category,
                'pair': pair,
                'layer': int(layer),
                'cosine_sim': sim,
            })

# ── Plot 1: Mean cosine similarity by layer ───────────────────────────────────
fig4 = make_subplots(
    rows=1, cols=len(categories),
    subplot_titles=[cat.capitalize() for cat in categories],
    shared_yaxes=True,
)

for col_idx, cat in enumerate(categories, start=1):
    cat_rows = [r for r in sim_rows if r['category'] == cat]
    for pair in pair_labels:
        pair_rows = [r for r in cat_rows if r['pair'] == pair]
        layer_to_sims = {}
        for r in pair_rows:
            layer_to_sims.setdefault(r['layer'], []).append(r['cosine_sim'])
        layers_sorted = sorted(layer_to_sims.keys())
        mean_sims = [np.mean(layer_to_sims[l]) for l in layers_sorted]
        fig4.add_trace(
            go.Scatter(
                x=layers_sorted, y=mean_sims,
                mode='lines', name=pair_labels[pair],
                line=dict(color=pair_colors[pair]),
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )
    # Threshold line
    fig4.add_hline(
        y=SIMILARITY_THRESHOLD, line_dash='dash', line_color='gray',
        annotation_text=f'{SIMILARITY_THRESHOLD}', row=1, col=col_idx,
    )

fig4.update_layout(
    title='Mean Cosine Similarity of Residual Streams by Layer',
    height=400,
)
fig4.update_yaxes(title_text='Cosine similarity', range=[0, 1.05], row=1, col=1)
fig4.update_xaxes(title_text='Layer')
fig4.show()

# ── Plot 2: Layer where similarity first crosses 0.9 ─────────────────────────
print(f"\nLayer where mean cosine similarity first crosses {SIMILARITY_THRESHOLD}:")
print(f"{'Category':<12} {'Pair':<25} {'First layer >= threshold':>25}")
print("-" * 65)
for cat in categories:
    for pair in pair_labels:
        cat_pair_rows = [r for r in sim_rows if r['category'] == cat and r['pair'] == pair]
        if not cat_pair_rows:
            continue
        layer_to_sims = {}
        for r in cat_pair_rows:
            layer_to_sims.setdefault(r['layer'], []).append(r['cosine_sim'])
        crossing_layer = None
        for l in sorted(layer_to_sims.keys()):
            if np.mean(layer_to_sims[l]) >= SIMILARITY_THRESHOLD:
                crossing_layer = l
                break
        val = str(crossing_layer) if crossing_layer is not None else 'never'
        print(f"{cat:<12} {pair_labels[pair]:<25} {val:>25}")

# ── Plot 3: Heatmap — mean cosine sim (layer × category, per pair) ────────────
# One heatmap per language pair
for pair in pair_labels:
    pair_rows = [r for r in sim_rows if r['pair'] == pair]
    # Build matrix: rows = categories, cols = layers
    max_layer = max(r['layer'] for r in pair_rows) if pair_rows else 0
    z = []
    for cat in categories:
        row_vals = []
        for l in range(max_layer + 1):
            vals = [r['cosine_sim'] for r in pair_rows if r['category'] == cat and r['layer'] == l]
            row_vals.append(np.mean(vals) if vals else np.nan)
        z.append(row_vals)

    fig_hm = go.Figure(go.Heatmap(
        z=z,
        x=list(range(max_layer + 1)),
        y=categories,
        colorscale='RdYlGn',
        zmin=0, zmax=1,
        colorbar=dict(title='Cosine sim'),
    ))
    fig_hm.update_layout(
        title=f'Cosine Similarity Heatmap: {pair_labels[pair]}',
        xaxis_title='Layer',
        yaxis_title='Category',
        height=300,
    )
    fig_hm.show()

## Cell 10 — Summary & Cross-Reference with step1.py

Cross-reference mechanistic findings with the accuracy results from `step1.py`:
- Do questions where Basque accuracy is low show later convergence in logit lens?
- Do math questions converge earlier (more language-agnostic)?

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 70)
print("SUMMARY: Logit Lens — Mean first-correct layer by category & language")
print("=" * 70)
print(f"{'Category':<12}", end="")
for lang in LANGUAGES:
    print(f"{lang.capitalize():>14}", end="")
print()
print("-" * 54)

for cat in categories:
    print(f"{cat:<12}", end="")
    for lang in LANGUAGES:
        fc_vals = [
            first_correct[qid].get(lang)
            for qid in logit_lens_results
            if triplets[qid]['english']['category'] == cat
            and first_correct[qid].get(lang) is not None
        ]
        val = f"{np.mean(fc_vals):.1f}" if fc_vals else "N/A"
        print(f"{val:>14}", end="")
    print()

print()
print("=" * 70)
print("SUMMARY: Representation Similarity — Mean cosine sim at last layer")
print("=" * 70)
last_layer = model.cfg.n_layers - 1
print(f"{'Category':<12}", end="")
for pair in pair_labels:
    print(f"{pair_labels[pair]:>28}", end="")
print()
print("-" * 96)

for cat in categories:
    print(f"{cat:<12}", end="")
    for pair in pair_labels:
        vals = [
            sim_results[qid][pair].get(last_layer, sim_results[qid][pair].get(str(last_layer)))
            for qid in sim_results
            if triplets[qid]['english']['category'] == cat
        ]
        vals = [v for v in vals if v is not None]
        val = f"{np.mean(vals):.3f}" if vals else "N/A"
        print(f"{val:>28}", end="")
    print()

# ── Cross-reference with step1.py accuracy ───────────────────────────────────
print()
print("=" * 70)
print("CROSS-REFERENCE NOTES")
print("=" * 70)
print("""
To cross-reference with step1.py accuracy results:

1. Run step1.py to generate per-question accuracy data, or load a
   pre-saved results JSON (add --save-results to step1.py if needed).

2. For each question id where Basque accuracy = 0 (model incorrect),
   check first_correct[qid]['basque']:
   - Late or None → model fails mechanistically (never identifies answer)
   - Early → model identifies answer internally but output fails (decoding?)

3. Compare mean first-correct layer for math vs reasoning:
   - Math expected to converge earlier (numerical answers are language-agnostic)
   - Reasoning expected to show larger English–Basque gap

4. Check English↔Basque cosine similarity at layer 0 vs last layer:
   - If late-layer similarity is high (>0.9), the model builds shared
     representations regardless of language — accuracy drops are due to
     earlier layers or output decoding, not final representation quality.
""")

print("Analysis complete. Results saved to:")
print("  logit_lens_results.json")
print("  sim_results.json")